In [112]:
import pandas as pd
import numpy as np
import os


In [113]:
# ==============================================================================
# BƯỚC 0: TẢI VÀ KIỂM TRA DỮ LIỆU GỐC (Load & Validate Raw Data)
# ==============================================================================

# 1. Khai báo đường dẫn file 
# (Luôn gán đường dẫn vào một biến để dễ dàng thay đổi ở một nơi duy nhất)
file_path = "../data/data_raw.csv"

# 2. Kiểm tra sự tồn tại của file trước khi đọc
if os.path.exists(file_path):
    print(f"[THÀNH CÔNG] Đã tìm thấy file dữ liệu tại: {file_path}")
    
    # Đọc file CSV và lưu vào biến df_raw
    df_raw = pd.read_csv(file_path)
    
    # In ra báo cáo nhanh về số lượng dữ liệu
    print(f"Kích thước dữ liệu gốc: {df_raw.shape[0]} dòng, {df_raw.shape[1]} cột.\n")
    
    # Xem trước 5 dòng đầu tiên để đối chiếu định dạng các cột
    display(df_raw.head(5)) 

else:
    # Cảnh báo rõ ràng giúp người dùng biết chính xác lỗi ở đâu thay vì để Python văng lỗi đỏ lòm
    print(f"[LỖI TỚI MÁY!] Không tìm thấy file dữ liệu tại đường dẫn: '{file_path}'")
    print("Vui lòng kiểm tra lại xem file đã được đặt đúng thư mục '../data/' chưa.")

[THÀNH CÔNG] Đã tìm thấy file dữ liệu tại: ../data/data_raw.csv
Kích thước dữ liệu gốc: 7043 dòng, 33 cột.



,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [114]:
# ==============================================================================
# BƯỚC 1: LỌC DỮ LIỆU (Giữ lại các cột cần thiết theo từng nhóm chức năng)
# ==============================================================================
columns_to_keep = [
    # 1.1. Biến định danh (Identification)
    'CustomerID',
    
    # 1.2. Biến mục tiêu (Target Variable)
    'Churn Label', 
    
    # 1.3. Nhóm Nhân khẩu học (Demographic Features)
    'Senior Citizen', 
    'Partner', 
    'Dependents', 
    
    # 1.4. Nhóm Dịch vụ sử dụng (Service Features)
    'Internet Service', 
    'Online Security', 
    'Online Backup', 
    'Device Protection', 
    'Tech Support', 
    
    # 1.5. Nhóm Hợp đồng & Cước phí (Contract & Billing Features)
    'Tenure Months', 
    'Contract', 
    'Payment Method', 
    'Paperless Billing', # Cột bản lề về hành vi thanh toán
    'Monthly Charges', 
    'Total Charges'
]

# Lọc dữ liệu từ df gốc
df = df_raw[columns_to_keep].copy()

# ==============================================================================
# BƯỚC 2: CHUẨN HÓA TÊN CỘT (Xóa khoảng trắng, đặt tên ngắn gọn)
# ==============================================================================
rename_mapping = {
    'CustomerID': 'ID',
    'Churn Label': 'Churn',
    'Tenure Months': 'Tenure',
    'Senior Citizen': 'SeniorCitizen',
    'Internet Service': 'InternetService',
    'Online Security': 'OnlineSecurity',
    'Online Backup': 'OnlineBackup',
    'Device Protection': 'DeviceProtection',
    'Tech Support': 'TechSupport',
    'Payment Method': 'PaymentMethod',
    'Monthly Charges': 'MonthlyCharges',
    'Total Charges': 'TotalCharges',
    'Paperless Billing': 'PaperlessBilling'
}

df.rename(columns=rename_mapping, inplace=True)

# ==============================================================================
# BƯỚC 3: KIỂM TRA KẾT QUẢ THEO TỪNG NHÓM CHỨC NĂNG
# ==============================================================================
print(f"Kích thước DataFrame sau khi lọc: {df.shape[0]} dòng, {df.shape[1]} cột\n")

print("DANH SÁCH CỘT ĐÃ CHUẨN HÓA (Đã chia nhóm chức năng):")
print("-" * 60)
print(f"{'Biến Định Danh':<25} : {['ID']}")
print(f"{'Biến Mục Tiêu':<25} : {['Churn']}")
print(f"{'Nhóm Nhân Khẩu Học':<25} : {['SeniorCitizen', 'Partner', 'Dependents']}")
print(f"{'Nhóm Dịch Vụ':<25} : {['InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']}")
print(f"{'Nhóm Hợp Đồng & Cước phí':<25} : {['Tenure', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges']}")

Kích thước DataFrame sau khi lọc: 7043 dòng, 16 cột

DANH SÁCH CỘT ĐÃ CHUẨN HÓA (Đã chia nhóm chức năng):
------------------------------------------------------------
Biến Định Danh            : ['ID']
Biến Mục Tiêu             : ['Churn']
Nhóm Nhân Khẩu Học        : ['SeniorCitizen', 'Partner', 'Dependents']
Nhóm Dịch Vụ              : ['InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
Nhóm Hợp Đồng & Cước phí  : ['Tenure', 'Contract', 'PaymentMethod', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges']


## ---------------- DATA CLEANNING ----------------

## 💡 Quy Trình (Workflow) Làm Sạch Dữ Liệu "Bất Bại"

Hãy kết hợp các phương pháp trên theo đúng thứ tự này để code của bạn luôn gọn gàng, chuẩn xác và không bỏ sót lỗi:

### Bước 1: Quét dọn toàn cục
Dọn dẹp các giá trị rác như khoảng trắng, dấu `?`, chuỗi `N/A` ở **tất cả các cột** (bao gồm cả cột chữ và cột số), sau đó ép toàn bộ chúng về định dạng chuẩn `NaN`.

### Bước 2: Xử lý cột chữ (Categorical)
Đồng nhất dữ liệu chữ bằng cách đưa tất cả về dạng viết thường thông qua hàm `.str.lower()`. Sau đó, sử dụng `.map()` để ánh xạ thành số (giống như cách bạn đã làm xuất sắc với cột `Contract`).

### Bước 3: Xử lý cột số (Numerical)
Áp dụng hàm `pd.to_numeric(errors='coerce')` cho các cột mang tính toán. Thuộc tính `coerce` sẽ đóng vai trò như một màng lọc cuối cùng, triệt tiêu nốt các ký tự lạ lọt lưới và tự động biến chúng thành `NaN`.

### Bước 4: Chốt hạ (Xử lý Missing Values)
Sử dụng lệnh `df.isnull().sum()` để thống kê xem có bao nhiêu `NaN` vừa xuất hiện sau 3 bước trên. Dựa vào đặc thù của từng cột dữ liệu để đưa ra quyết định cuối cùng:
* Điền số `0` bằng lệnh `.fillna(0)`.
* Điền giá trị trung bình/trung vị bằng `.fillna(df['...'].mean())`.
* Xóa hoàn toàn các dòng chứa lỗi bằng `.dropna()`.

In [115]:
# Clean data cơ bản chuyển tất cả các ô có kí tự lạ về Nan
# 1. Dùng Regex để tìm mọi ô CHỈ CÓ khoảng trắng (1 hoặc nhiều dấu cách) và biến thành NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# 2. Thay thế một danh sách các ký tự lạ/chuỗi vô nghĩa cụ thể thành NaN
list_garbage = ['?', '-', 'N/A', 'NA', 'null', 'None']
df = df.replace(list_garbage, np.nan)

In [116]:
# ['OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','PaperlessBilling','Partner' ]
# Xử lý các biến boolean
def encode_binary_columns(df, columns):
    # Bước 1: Chuẩn hóa chuỗi (in thường, xóa khoảng trắng thừa) cho các cột là Text
    for col in columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip().str.lower()
            
    # Bước 2: Thay thế hàng loạt
    mapping_dict = {
        'yes': 1, 
        'no': 0, 
        'no internet service': 0 # Cột nào không có chữ này Pandas sẽ tự động bỏ qua
    }
    df[columns] = df[columns].replace(mapping_dict)
    
    return df


cols_to_encode = [
    'OnlineSecurity', 
    'OnlineBackup', 
    'DeviceProtection', 
    'TechSupport', 
    'PaperlessBilling', 
    'Partner'
]

df = encode_binary_columns(df, cols_to_encode)

C:\Users\HGB\AppData\Local\Temp\ipykernel_3884\72833281.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[columns] = df[columns].replace(mapping_dict)


In [117]:
#['TotalCharges','Contract','Churn']
# Hàm xử lý các cột chữ (Categorical)
def convert_contract_variables(df,columns):
    # Xử lý Contract ('month-to-month' -> 1, 'one year'-> 12, 'two year'-> 24)
    mapping_contract = {'month-to-month': 1, 'one year': 12, 'two year': 24}
    for col in columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip().str.lower()
            # SỬA: Đưa map() vào bên trong vòng lặp for và if
            df[col] = df[col].map(mapping_contract)
    
    return df

def convert_churn_variables(df,columns):
    # Xử lý Churn  (Yes -> 1, No -> 0)
    mapping_churn = {'yes': 1, 'no': 0}

    for col in columns:
        if df[col].dtype == 'object':
            df[col] = df[col].str.strip().str.lower()
            # SỬA: Đưa map() vào bên trong vòng lặp for và if
            df[col] = df[col].map(mapping_churn)
    
    return df

def convert_to_float(df, columns):
    for col in columns:
        if df[col].dtype == 'object':
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Xử lý cho TotalCharges
df = convert_to_float(df, ['TotalCharges'])

# Xử lý cho Churn, Contract
df = convert_churn_variables(df, ['Churn'])

df = convert_contract_variables(df, ['Contract']) 

In [118]:
# 4.1. Kiểm tra số lượng giá trị khuyết thiếu trên các cột được chỉ định
cols_to_check = [
    'ID', 'Churn', 'SeniorCitizen', 'Partner', 'Dependents', 'Tenure', 'Contract', 
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 
    'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'PaperlessBilling'
]

print("Số dòng bị lỗi/trống:\n")
print(df[cols_to_check].isnull().sum())

# 4.2. Xử lý khoảng trống cho TotalCharges
# Business Logic: Khách hàng mới (Tenure = 0) chưa phát sinh cước nên TotalCharges bị NaN. 
# Xử lý bằng cách điền 0.
df['TotalCharges'] = df['TotalCharges'].fillna(0)

Số dòng bị lỗi/trống:

ID                   0
Churn                0
SeniorCitizen        0
Partner              0
Dependents           0
Tenure               0
Contract             0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
PaperlessBilling     0
dtype: int64


In [120]:
df.describe()

,Churn,Partner,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,Tenure,Contract,PaperlessBilling,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.265370,0.483033,0.286668,0.344881,0.343888,0.290217,32.371149,8.835865,0.592219,64.761692,2279.734304
std,0.441561,0.499748,0.452237,0.475363,0.475038,0.453895,24.559481,9.551444,0.491457,30.090047,2266.794470
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,18.250000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,1.000000,0.000000,35.500000,398.550000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,29.000000,1.000000,1.000000,70.350000,1394.550000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,55.000000,12.000000,1.000000,89.850000,3786.600000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,72.000000,24.000000,1.000000,118.750000,8684.800000
